# Session 1: a lift goes up and down
**A. Real-world problem.** You step into a lift and the floor seems to push harder on your feet for a
moment as it starts to rise, and softer when it starts to go down. Nothing about you changed. What did?

**Plan.** (1) draw the speed-time graph of a ride, (2) find the acceleration from its slope and the
distance from its area, (3) work out the force of the floor on your feet (the "apparent weight").

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

G = 9.81   # m/s^2, acceleration of gravity

## Step 1: the speed-time graph of a ride
A simple ride has three parts: **accelerate** (speed rises in a straight line), **cruise** (speed is
constant), **decelerate** (speed falls to zero). The graph is a trapezoid.
- The **slope** of the graph is the acceleration a (m/s^2).
- The **area** under the graph is the distance travelled (m).

With cruise speed v and acceleration a, speeding up takes v / a seconds and covers v^2 / (2a) metres.
So for a total distance d (if the lift reaches the cruise speed): cruise distance = d - v^2 / a.

In [ ]:
def ride_times(d, v, a):
    """Times (accelerate, cruise, decelerate) in seconds for distance d (m), top speed v (m/s), acceleration a (m/s^2).

    If the distance is too short to reach v, the lift speeds up and then at once slows down (cruise time 0)
    and the top speed is smaller than v.
    """
    if d >= v ** 2 / a:
        t_acc = v / a
        return t_acc, (d - v ** 2 / a) / v, t_acc
    t_acc = np.sqrt(d / a)
    return t_acc, 0.0, t_acc


def speed_at(t, d, v, a):
    """Speed (m/s) at time t (s) during a ride going up."""
    t_acc, t_cruise, t_dec = ride_times(d, v, a)
    top = a * t_acc
    total = t_acc + t_cruise + t_dec
    if t < 0 or t > total:
        return 0.0
    if t < t_acc:
        return a * t
    if t < t_acc + t_cruise:
        return top
    return top - a * (t - t_acc - t_cruise)


def travel_time(d, v, a):
    return sum(ride_times(d, v, a))

## Step 2: why you feel heavier
You feel the force of the floor on you. When the lift is not accelerating that force equals your
weight m g. When the lift accelerates upward with a, the floor must push you up with more force so
that you accelerate too (Newton's second law: net force = m a):

    N - m g = m a        so        N = m (g + a)

Going up and speeding up: a > 0, you feel heavier. Going up and slowing down: the acceleration points
down (a < 0), you feel lighter. A lift in free fall (a = -g) would give N = 0.

In [ ]:
def apparent_weight(m, a):
    """Force of the floor on a person of mass m (kg) when the lift has acceleration a (m/s^2, up positive), in newtons."""
    return m * (G + a)


def feel_ratio(a):
    """Apparent weight divided by true weight: 1 + a / g (no unit)."""
    return 1.0 + a / G

## Step 3: see it
Left: speed against time (a trapezoid). Right: the floor force on a 60 kg person during the same ride.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    d, v, a, m = 21.0, 2.0, 1.0, 60.0
    T = travel_time(d, v, a)
    t = np.linspace(0, T, 400)
    speed = np.array([speed_at(x, d, v, a) for x in t])
    t_acc, t_cr, t_dec = ride_times(d, v, a)
    acc = np.where(t < t_acc, a, np.where(t < t_acc + t_cr, 0.0, -a))
    print(f"ride of {d} m at {v} m/s: accelerate {t_acc:.1f} s, cruise {t_cr:.1f} s, decelerate {t_dec:.1f} s, total {T:.1f} s")
    print(f"weight at rest {m * G:.0f} N, while speeding up {apparent_weight(m, a):.0f} N, "
          f"while slowing down {apparent_weight(m, -a):.0f} N")

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(t, speed, color="tab:blue")
    ax.fill_between(t, speed, alpha=0.15)
    ax.set_xlabel("time (s)")
    ax.set_ylabel("speed (m/s)")
    ax.set_title("Speed-time graph: area = distance (21 m)")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "speed_time.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(t, [apparent_weight(m, x) for x in acc], color="tab:red")
    ax2.axhline(m * G, color="gray", ls=":", label="true weight 589 N")
    ax2.set_xlabel("time (s)")
    ax2.set_ylabel("floor force on a 60 kg person (N)")
    ax2.set_title("Heavier when speeding up, lighter when slowing down")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "apparent_weight.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the checkpoint.

## Coding tasks
1. **Easy.** Change `d, v, a` in `lesson.py` to a 10 m ride at 2 m/s with 1 m/s^2. Predict the three times (2, 3, 2 s) before you run it.
2. **Stretch.** Write `floor_force_profile(m, d, v, a)` that returns the floor force during each of the three parts of the ride, and print it for m = 80 kg.
3. **Explain.** Explain why the floor force is the same during a 5 m/s cruise as during a 0.5 m/s cruise.

## Assessment
- **A. Conceptual.** A lift is moving downward at a steady speed and then begins to slow down to stop at the ground floor. Do you feel heavier, lighter or the same? Explain without using the word speed.
- **B. Numerical.** A 75 kg person stands in a lift that speeds up upward at 1.5 m/s^2. Find the floor force and the percentage change from the person's weight at rest.
- **C. Graph and simulation interpretation.** On the speed-time graph of the 21 m ride (top speed 2 m/s, rise over 2 s, flat for 8.5 s), what is the acceleration and how can you check that the distance is 21 m?
- **D. Design and optimisation challenge.** Design: the 21 m ride must take less than 11 s and the floor force must stay within 15 percent of the person's weight (a <= 1.47 m/s^2). Choose a and v, check d >= v^2 / a, and give the ride time. Why not simply choose a very large v?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("hand calculation: 10 m at 2 m/s with a = 1 m/s^2 takes 2 + 3 + 2 = 7 s",
                   np.allclose(L.ride_times(10, 2, 1), (2.0, 3.0, 2.0)) and abs(L.travel_time(10, 2, 1) - 7.0) < 1e-12))
    N_up = L.apparent_weight(60.0, 1.0)
    N_rest = L.apparent_weight(60.0, 0.0)
    checks.append(("worked example: a 60 kg person speeding up at 1.0 m/s^2 feels 648.6 N (588.6 N at rest)",
                   abs(N_up - 648.6) < 0.05 and abs(N_rest - 588.6) < 0.05))
    t = np.linspace(0, L.travel_time(21, 2, 1), 20001)
    v = np.array([L.speed_at(x, 21, 2, 1) for x in t])
    area = float(np.sum(0.5 * (v[1:] + v[:-1]) * np.diff(t)))
    checks.append(("the area under the speed-time graph equals the distance (21 m)", abs(area - 21.0) < 0.01))
    checks.append(("slowing down at 1 m/s^2 makes a 60 kg person feel lighter than at rest",
                   L.apparent_weight(60.0, -1.0) < L.apparent_weight(60.0, 0.0)))
    checks.append(("free fall (a = -g) gives zero floor force", abs(L.apparent_weight(60.0, -L.G)) < 1e-9))
    t1, tc, t2 = L.ride_times(3.0, 2.0, 1.0)
    top = 1.0 * t1
    checks.append(("short ride: no cruise, the top speed is lower than the limit (sqrt(3) = 1.73 m/s)",
                   tc == 0.0 and abs(top - np.sqrt(3.0)) < 1e-9 and top < 2.0))
    checks.append(("boundary: d = v^2/a exactly gives zero cruise time and no error",
                   abs(L.ride_times(4.0, 2.0, 1.0)[1]) < 1e-12))
    checks.append(("feel ratio 1 + a/g is 1 at rest and above 1 when speeding up",
                   L.feel_ratio(0.0) == 1.0 and L.feel_ratio(1.0) > 1.0))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
